# Занятие 3: Генеративные модели, правдоподобие и предсказание
### Рабочая тетрадь студента
*Курс: Байесовский анализ эмпирических данных (2026)*

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/03_generative_models_likelihood_prediction_ru.ipynb)

English version: [03_generative_models_likelihood_prediction.ipynb](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/03_generative_models_likelihood_prediction.ipynb)

## 1. Сделайте свою копию
Чтобы сохранять изменения, нажмите **Файл $\to$ Сохранить копию на Диске** (File → Save a copy in Drive). Можно также скачать блокнот как файл `.ipynb` и запускать локально.

## 2. Проверка окружения
Импортируем библиотеки и настраиваем Plotly для Google Colab.

In [1]:
import sys
import numpy as np
import pandas as pd
from scipy import stats
from scipy.integrate import trapezoid
import plotly.graph_objects as go
from plotly.subplots import make_subplots

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("⚡ Работаем в Google Colab.")
    import plotly.io as pio
    pio.renderers.default = "colab"
else:
    print("💻 Работаем в локальном окружении.")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
print(f"✅ Окружение готово. Зерно генератора NumPy: {RANDOM_SEED}.")

💻 Работаем в локальном окружении.
✅ Окружение готово. Зерно генератора NumPy: 42.


## 3. Учебные цели
В конце занятия хотелось бы:
1. Описать и запускать простую генеративную модель.
2. Отличать изменчивость данных при фиксированном параметре от неопределённости относительно самого параметра.
3. Находить неправдоподобные наблюдения и связывать их с допущением модели.
4. Различать априорное и апостериорное предсказательное моделирование.

## 4. Задание: предскажите до запуска
> ✍ **НАПИШИТЕ**:
> **A. Опрос.** В опросе $N = 20$ человек $k = 13$ поддержали некоторую меру.
> 1. При каком значении параметра поддержки $\theta \in [0, 1]$ наблюдаемые данные наиболее вероятны?
> 2. Будет ли интеграл функции правдоподобия $\mathcal{L}(\theta \mid k=13)$ по $\theta \in [0, 1]$ равен $1{,}0$? Почему?
>
> **B. Предсказание.** До всяких вычислений:
> 3. Если бы мы не фиксировали $\theta$, а сначала извлекали его из априорного распределения, симулированные числа были бы разбросаны сильнее или слабее, чем при фиксированном $\theta = 0{,}65$?
> 4. После наблюдения $k = 13$ предсказание для *следующего* опроса будет уже или шире, чем апостериорное распределение самого $\theta$?

## 5. воспроизведите
### A. Разобранный пример: симуляция опросов из известной модели
▶ **ЗАПУСКАЕМ ВМЕСТЕ**: начинаем в «генеративном мире». Пусть истинная доля поддержки $\theta_0 = 0{,}65$. Симулируем $S = 1000$ независимых опросных организаций, каждая опрашивает $N = 20$ человек.

In [2]:
N_trials = 20
theta_true = 0.65
S_sims = 1000

# Прямая симуляция: выборки из Binomial(N=20, theta=0.65)
simulated_k = rng.binomial(n=N_trials, p=theta_true, size=S_sims)

print(f"Симулировано {S_sims} повторений опроса:")
print(f"  Теоретическое среднее: {N_trials * theta_true:.2f}")
print(f"  Эмпирическое среднее:  {simulated_k.mean():.2f}")
print(f"  Теоретическое SD:      {np.sqrt(N_trials * theta_true * (1 - theta_true)):.2f}")
print(f"  Эмпирическое SD:       {simulated_k.std():.2f}")

counts, bins = np.histogram(simulated_k, bins=np.arange(-0.5, N_trials + 1.5, 1))
fig_sim = go.Figure(go.Bar(
    x=np.arange(0, N_trials + 1),
    y=counts / S_sims,
    marker_color='#2b6cb0',
    hovertemplate='<b>k = %{x}</b><br>Частота в симуляции: %{y:.3f}<extra></extra>'
))
fig_sim.update_layout(
    title=f'Прямая симуляция: Binomial(N={N_trials}, θ={theta_true}), {S_sims} повторений',
    xaxis_title='Число сторонников k (из 20)',
    yaxis_title='Относительная частота',
    template='plotly_white',
    height=380
)
fig_sim.show()

Симулировано 1000 повторений опроса:
  Теоретическое среднее: 13.00
  Эмпирическое среднее:  13.02
  Теоретическое SD:      2.13
  Эмпирическое SD:       2.15



### 🛑 Внимание вопрос
как меняется распределение выборочной доли $\hat{p} = k/N$ с ростом объёма выборки? Будет ли в каждом симулированном опросе ровно 13 сторонников?

## 6. Измените
### B. Правдоподобие: данные фиксированы, параметр меняется
🧪 **ИЗМЕНИТЕ ОДНО**: сначала измените объём выборки в прямой симуляции, затем зафиксируйте наблюдаемые данные и дайте меняться параметру.

In [3]:
# 🧪 ПЕСОЧНИЦА: прямая симуляция
# Измените параметры ниже и запустите, чтобы проверить свои гипотезы.

sandbox_N = 200        # Попробуйте: 20, 50, 200, 1000
sandbox_theta = 0.65   # Попробуйте: 0.04 (редкие события), 0.50 (поровну), 0.65 (большинство)
sandbox_sims = 1000

sandbox_k = rng.binomial(n=sandbox_N, p=sandbox_theta, size=sandbox_sims)
sandbox_prop = sandbox_k / sandbox_N

print(f"Результаты для N={sandbox_N}, θ={sandbox_theta}:")
print(f"  Средняя доля: {sandbox_prop.mean():.4f} (истинная: {sandbox_theta:.4f})")
print(f"  SD доли:      {sandbox_prop.std():.4f} (теоретическая SE: {np.sqrt(sandbox_theta*(1-sandbox_theta)/sandbox_N):.4f})")
print(f"  Min k: {sandbox_k.min()}, Max k: {sandbox_k.max()}")

Результаты для N=200, θ=0.65:
  Средняя доля: 0.6501 (истинная: 0.6500)
  SD доли:      0.0335 (теоретическая SE: 0.0337)
  Min k: 107, Max k: 149


In [4]:
k_obs = 13
N_obs = 20
theta_grid = np.linspace(0.001, 0.999, 500)

# Правдоподобие: L(theta | k=13, N=20) = binom.pmf(13, 20, theta)
likelihood = stats.binom.pmf(k_obs, N_obs, theta_grid)

mle_idx = np.argmax(likelihood)
mle_theta = theta_grid[mle_idx]

# Два конкурирующих значения параметра
h1_val = 0.65
h2_val = 0.50
l_h1 = stats.binom.pmf(k_obs, N_obs, h1_val)
l_h2 = stats.binom.pmf(k_obs, N_obs, h2_val)
lr_h1_h2 = l_h1 / l_h2

fig_lik = go.Figure()
fig_lik.add_trace(go.Scatter(
    x=theta_grid, y=likelihood, mode='lines',
    line=dict(color='#d97706', width=2.5), fill='tozeroy', fillcolor='rgba(217, 119, 6, 0.12)',
    name=f'Правдоподобие L(θ | k={k_obs})'
))
fig_lik.add_trace(go.Scatter(
    x=[h2_val, h1_val], y=[l_h2, l_h1], mode='markers+text',
    text=[f'H2: θ={h2_val}<br>(L={l_h2:.4f})', f'H1: θ={h1_val}<br>(L={l_h1:.4f})'],
    textposition=['bottom left', 'top right'],
    marker=dict(size=10, color=['#c53030', '#276749']), name='Кандидаты'
))
fig_lik.update_layout(
    title=f'Правдоподобие L(θ | k={k_obs}, N={N_obs})   [отношение правдоподобий H1/H2 = {lr_h1_h2:.2f}]',
    xaxis_title='Кандидат θ (вероятность поддержки)',
    yaxis_title='Правдоподобие L(θ)',
    template='plotly_white', height=420
)
fig_lik.show()

print(f"Оценка максимального правдоподобия: θ_hat = {mle_theta:.3f} (выборочная доля {k_obs}/{N_obs} = {k_obs/N_obs:.3f})")
print(f"Правдоподобие при H1 (θ={h1_val}): {l_h1:.4f}")
print(f"Правдоподобие при H2 (θ={h2_val}): {l_h2:.4f}")
print(f"Отношение правдоподобий: {lr_h1_h2:.3f} -> данные в {lr_h1_h2:.2f} раза лучше согласуются с H1, чем с H2.")

Оценка максимального правдоподобия: θ_hat = 0.649 (выборочная доля 13/20 = 0.650)
Правдоподобие при H1 (θ=0.65): 0.1844
Правдоподобие при H2 (θ=0.5): 0.0739
Отношение правдоподобий: 2.494 -> данные в 2.49 раза лучше согласуются с H1, чем с H2.


In [ ]:
# 🧪 ПЕСОЧНИЦА: свои сценарии для правдоподобия
k_sandbox = 65       # Попробуйте: 65 (большая выборка), 0 (ни одного события), 7
N_sandbox = 100      # Попробуйте: 100, 20, 10
hypo_1 = 0.65        # Попробуйте: 0.65, 0.80
hypo_2 = 0.50        # Попробуйте: 0.50, 0.40

lik_curve = stats.binom.pmf(k_sandbox, N_sandbox, theta_grid)
l_h1_sb = stats.binom.pmf(k_sandbox, N_sandbox, hypo_1)
l_h2_sb = stats.binom.pmf(k_sandbox, N_sandbox, hypo_2)
lr_sb = l_h1_sb / l_h2_sb if l_h2_sb > 0 else np.inf

print(f"Эксперимент в песочнице (k={k_sandbox}/{N_sandbox}):")
print(f"  ОМП:                          {theta_grid[np.argmax(lik_curve)]:.3f}")
print(f"  Правдоподобие при H1:         {l_h1_sb:.6e}")
print(f"  Правдоподобие при H2:         {l_h2_sb:.6e}")
print(f"  Отношение правдоподобий H1/H2: {lr_sb:.2f}x")

Эксперимент в песочнице (k=65/100):
  ОМП:                          0.649
  Правдоподобие при H1:         8.340469e-02
  Правдоподобие при H2:         8.638557e-04
  Отношение правдоподобий H1/H2: 96.55x



### 🛑 Правдаподобие не распределение
▶ **ЗАПУСКАЕМ ВМЕСТЕ**: покажем численно, почему правдоподобие — НЕ плотность, вычислив площадь под кривой.

In [6]:
area_lik = trapezoid(likelihood, theta_grid)
theoretical_area = 1.0 / (N_obs + 1)

print(f"Численная площадь под L(θ | k=13, N=20): {area_lik:.5f}")
print(f"Точная площадь 1/(N+1) = 1/21:           {theoretical_area:.5f}")
print(f"Является ли правдоподобие плотностью по θ? {'ДА' if np.isclose(area_lik, 1.0) else 'НЕТ (площадь ≠ 1.0)'}")

Численная площадь под L(θ | k=13, N=20): 0.04762
Точная площадь 1/(N+1) = 1/21:           0.04762
Является ли правдоподобие плотностью по θ? НЕТ (площадь ≠ 1.0)


## 7. априорное и апостериорное предсказание
###  Откуда берутся предсказания? (анонс занятия 4)
▶ **ЗАПУСКАЕМ ВМЕСТЕ**: до сих пор $\theta$ был фиксирован. Вместо фиксированного $\theta$ извлеките его из априорного распределения $\operatorname{Beta}(2, 2)$ перед генерацией чисел; затем повторите с апостериорным $\operatorname{Beta}(2 + 13, 2 + 7)$ после наблюдения $k = 13$. Так оба источника неопределённости видны одновременно.

In [7]:
# 1. Априорное: Beta(2, 2)
prior_a, prior_b = 2, 2
theta_prior_draws = rng.beta(prior_a, prior_b, size=2000)
y_prior_pred = rng.binomial(n=N_obs, p=theta_prior_draws)      # априорное предсказательное

# 2. Апостериорное: Beta(2 + 13, 2 + 7) = Beta(15, 9)
post_a = prior_a + k_obs
post_b = prior_b + (N_obs - k_obs)
theta_post_draws = rng.beta(post_a, post_b, size=2000)
y_post_pred = rng.binomial(n=N_obs, p=theta_post_draws)        # апостериорное предсказательное

fig_pred = make_subplots(rows=1, cols=2, subplot_titles=[
    '<b>Априорное предсказательное</b><br><span style="font-size:11px;color:#64748b">до наблюдения данных</span>',
    '<b>Апостериорное предсказательное</b><br><span style="font-size:11px;color:#64748b">после наблюдения k = 13</span>'])
fig_pred.add_trace(go.Histogram(x=y_prior_pred, histnorm='probability', marker_color='#94a3b8'), row=1, col=1)
fig_pred.add_trace(go.Histogram(x=y_post_pred, histnorm='probability', marker_color='#2563eb'), row=1, col=2)
fig_pred.add_vline(x=k_obs, line_dash='dash', line_color='#dc2626', annotation_text='наблюдение k = 13', row=1, col=2)
fig_pred.update_layout(template='plotly_white', height=420, showlegend=False)
fig_pred.update_xaxes(title_text='Симулированное k (из 20)', range=[-0.5, 20.5])
fig_pred.update_yaxes(title_text='Вероятность')
fig_pred.show()

print(f"Среднее априорного предсказания:     {y_prior_pred.mean():.2f} (SD = {y_prior_pred.std():.2f})")
print(f"Среднее апостериорного предсказания: {y_post_pred.mean():.2f} (SD = {y_post_pred.std():.2f})")
print(f"Апостериорное среднее параметра:     {theta_post_draws.mean():.4f} (SD = {theta_post_draws.std():.4f})")

Среднее априорного предсказания:     10.10 (SD = 4.91)
Среднее апостериорного предсказания: 12.45 (SD = 2.88)
Апостериорное среднее параметра:     0.6224 (SD = 0.0967)


> ✍ **НАПИШИТЕ**: почему апостериорное предсказательное распределение нового результата опроса имеет больший разброс, чем апостериорное распределение самого $\theta$?

### 🛑 Внимание вопрос
1. Согласована ли симуляция с теоретическими ожиданиями?
2. Априорное предсказание заметно шире апостериорного. Какая часть этой ширины — неопределённость относительно $\theta$, а какая — разброс между опросами?
3. Опрос большего объёма делает предсказанную *долю* устойчивее. Делает ли он предсказуемее ответ отдельного человека?

## 7. Итоговая запись
> ✍ **НАПИШИТЕ**:
> 1. **Оцениваемая величина**: какую величину описывало правдоподобие в классной работе 2 и что было зафиксировано при его вычислении?
> 2. **Правдоподобие vs плотность**: своими словами — почему правдоподобие не является распределением вероятностей по параметру?
> 3. **Два вида неопределённости**: приведите по одному сегодняшнему примеру изменчивости при фиксированном параметре и неопределённости относительно параметра.
> 4. **Одно ограничение**: назовите одно допущение модели опроса, которое вы проверили бы на реальных данных первым.

## 8. Почти пуассон
###  счётные данные, которых Пуассон не порождает
До сих пор исход был ограничен сверху числом $N$. Но многие социальные исходы — неограниченные счётчики: число обращений в городскую горячую линию за день, число поданных жалоб, число посещённых мероприятий. Естественная первая модель — Пуассон, а он жёстко требует **дисперсия = среднее**.

Реальные счётчики этому почти никогда не подчиняются. Одна из генеративных историй: у каждого дня своя интенсивность $\lambda_t$ вокруг базового уровня (паттерн дня недели × случайная вариация от дня ко дню), а число событий при данном $\lambda_t$ — пуассоновское. Маргинально это отрицательное биномиальное распределение. Симулируйте 365 дней и посмотрите, как индекс дисперсии (дисперсия ÷ среднее) выходит за пределы того, что даёт настоящий пуассоновский процесс.

In [9]:
baseline_rate = 8.0                 # ожидаемое число событий в день
n_days = 365
alpha_disp = 4.0                    # Попробуйте: 0.5 (сильная сверхдисперсия), 4 (умеренная), 50 (почти Пуассон)
weekday_mult = np.array([0.9, 0.9, 0.95, 1.0, 1.2, 1.3, 0.9])   # Пн..Вс; поставьте все единицы, чтобы убрать паттерн


def dispersion_index(v):
    return v.var(ddof=1) / v.mean()


# Каким бывает индекс, когда счётчики ДЕЙСТВИТЕЛЬНО пуассоновские?
ref = np.array([dispersion_index(rng.poisson(lam=baseline_rate, size=n_days)) for _ in range(500)])
lo, hi = np.percentile(ref, [2.5, 97.5])

weekday = np.arange(n_days) % 7
lam_day = baseline_rate * weekday_mult[weekday] * rng.gamma(shape=alpha_disp, scale=1.0 / alpha_disp, size=n_days)
y_year = rng.poisson(lam=lam_day)

print(f"Год симулированных дневных счётчиков (alpha = {alpha_disp}):")
print(f"  среднее {y_year.mean():.2f}, дисперсия {y_year.var(ddof=1):.2f}, индекс дисперсии {dispersion_index(y_year):.2f}")
print(f"  настоящий пуассоновский процесс за {n_days} дней даёт индекс в [{lo:.2f}, {hi:.2f}]")
print(f"  среднее по дням недели (Пн..Вс): {np.round([y_year[weekday == d].mean() for d in range(7)], 2)}")

fig_hw = go.Figure()
fig_hw.add_trace(go.Histogram(x=rng.poisson(lam=baseline_rate, size=n_days), xbins=dict(size=1), histnorm='probability',
                              opacity=0.6, marker_color='#2b6cb0', name=f'Пуассон({baseline_rate:.0f})'))
fig_hw.add_trace(go.Histogram(x=y_year, xbins=dict(size=1), histnorm='probability',
                              opacity=0.6, marker_color='#e53e3e', name=f'День недели × Гамма-Пуассон (α={alpha_disp})'))
fig_hw.update_layout(template='plotly_white', height=380, barmode='overlay',
                     xaxis_title='событий в день', yaxis_title='Относительная частота')
fig_hw.show()

Год симулированных дневных счётчиков (alpha = 4.0):
  среднее 7.91, дисперсия 22.09, индекс дисперсии 2.79
  настоящий пуассоновский процесс за 365 дней даёт индекс в [0.85, 1.15]
  среднее по дням недели (Пн..Вс): [7.38 7.19 7.81 7.58 8.77 9.29 7.37]


## 9. Воспроизводимость
▶ **ЗАПУСКАЕМ ВМЕСТЕ**: запустите ячейку, чтобы зафиксировать параметры окружения.

In [10]:
import datetime
import scipy
import plotly
print(f"Время запуска: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Plotly: {plotly.__version__}")
print(f"Зерно генератора: {RANDOM_SEED}")
print("Данные: синтетические, сгенерированы в этой тетради (ничего не скачивается)")

Время запуска: 2026-09-22 18:24:38
Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 17:06:14) [Clang 19.1.7 ]
NumPy: 2.4.3
Pandas: 3.0.2
SciPy: 1.17.1
Plotly: 7.0.0
Зерно генератора: 42
Данные: синтетические, сгенерированы в этой тетради (ничего не скачивается)
